# CNN Warm-Up: Teaching a Computer to Recognize Handwritten Digits

Welcome! Before we build a reinforcement learning agent that plays **Pong** from raw pixels, let's get comfortable with the piece of technology that makes that possible: the **Convolutional Neural Network (CNN)**.

A CNN is the part of the system that "looks" at an image and figures out what's in it. In the Pong project, a CNN will look at the game screen and learn to recognize things like *where the ball is* and *where the paddles are*. Here, we'll use a much simpler and friendlier example: recognizing handwritten digits (0–9).

**By the end of this notebook you will understand:**
1. That an image is really just a grid of numbers.
2. What a "filter" (kernel) does to an image.
3. How a CNN is built out of these filters, stacked in layers.
4. How the network *learns* the right filters by training on examples.
5. How to peek inside a trained network to see what it learned.

No prior deep learning knowledge needed — just run each cell from top to bottom and read the explanations in between.


## Step 1 — An image is just numbers

A computer doesn't see a "5" the way we do. It sees a grid of numbers, where each number is the brightness of one pixel (0 = black, higher = brighter).

Let's load a small dataset of handwritten digits and look at a few of them — both as pictures and as raw numbers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
print("Number of images:", len(digits.images))
print("Each image size:", digits.images[0].shape, "pixels")
print("Possible labels (digits):", np.unique(digits.target))

# show the first 10 digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap="gray")
    ax.set_title(f"label: {digits.target[i]}")
    ax.axis("off")
plt.suptitle("A handful of digits from our dataset")
plt.tight_layout()
plt.show()


In [ ]:
# Let's look at ONE digit as raw numbers, side by side with the picture
sample = digits.images[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sample, cmap="gray")
axes[0].set_title(f"Picture (label = {digits.target[0]})")
axes[0].axis("off")

axes[1].imshow(sample, cmap="gray")
axes[1].set_title("Same image, but it's really just numbers")
for i in range(sample.shape[0]):
    for j in range(sample.shape[1]):
        axes[1].text(j, i, f"{int(sample[i, j])}", ha="center", va="center",
                     color="red", fontsize=8)
axes[1].axis("off")
plt.tight_layout()
plt.show()


## Step 2 — What does a "filter" (kernel) do?

A CNN scans a small window (e.g. 3x3 pixels) across the image. At each position, it multiplies the pixels under the window by a small grid of numbers called a **filter** (or **kernel**), and adds up the result. This produces one number in a new, smaller grid called a **feature map**.

Different filters detect different things — edges, corners, textures, etc. Let's try a classic hand-designed **edge-detection filter** and see what it does to one of our digits.


In [ ]:
edge_kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1],
], dtype=np.float32)

def manual_convolve(img, kernel):
    """Slide `kernel` over `img` and compute the convolution output by hand,
    so we can see exactly what a CNN filter does under the hood."""
    kh, kw = kernel.shape
    h, w = img.shape
    out = np.zeros((h - kh + 1, w - kw + 1), dtype=np.float32)
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            region = img[i:i + kh, j:j + kw]
            out[i, j] = np.sum(region * kernel)
    return out

sample = digits.images[0]
feature_map = manual_convolve(sample, edge_kernel)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(sample, cmap="gray")
axes[0].set_title("Original digit")
axes[0].axis("off")

axes[1].imshow(edge_kernel, cmap="RdBu")
axes[1].set_title("Edge-detection filter\n(3x3 numbers)")
axes[1].axis("off")

axes[2].imshow(feature_map, cmap="gray")
axes[2].set_title("Result: a 'feature map'\nhighlighting edges")
axes[2].axis("off")
plt.tight_layout()
plt.show()
print("Notice how the outline of the digit lights up: this filter reacts strongly to edges.")


**Key idea:** a hand-designed filter like this can highlight edges. But how do we know which filters are *useful* for recognizing digits? We don't have to guess — a CNN **learns** the best filters automatically from examples, by adjusting the numbers in the filters during training. That's exactly what we'll do next.


## Step 3 — Building a tiny CNN

Our network will have this structure:

1. **Conv layer 1**: 8 learnable filters (like the edge filter above, but starting random) scan the image and produce 8 feature maps.
2. **Pooling**: shrink each feature map by keeping only the strongest signal in each small region (this makes the network faster and more robust to small shifts).
3. **Conv layer 2**: 16 more filters scan the *feature maps* from layer 1, combining simple features into more complex ones.
4. **Pooling** again.
5. **Fully connected layers**: take all the extracted features and decide: "this looks most like a 7" (or whichever digit).

This is a *tiny* version of the same architecture used in the famous 2015 DeepMind Atari paper, where a CNN reads the raw pixels of a video game screen.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 2 * 2, 32)
        self.fc2 = nn.Linear(32, 10)  # 10 possible digits

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 8x8 image -> 4x4 feature maps
        x = self.pool(F.relu(self.conv2(x)))   # 4x4 -> 2x2 feature maps
        x = x.view(x.size(0), -1)              # flatten into a single vector
        x = F.relu(self.fc1(x))
        x = self.fc2(x)                        # 10 scores, one per digit
        return x

model = TinyCNN()
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal learnable numbers in this tiny network: {n_params}")


## Step 4 — Training: learning from examples

We show the network thousands of (image, correct label) pairs. Each time, it:
1. Makes a guess.
2. Compares its guess to the correct answer (the **loss** — how wrong it was).
3. Slightly adjusts every filter number to be a little less wrong next time (this adjustment step is called **gradient descent**).

Repeating this many times (each full pass over the data is called an **epoch**) is how the network gradually learns good filters — without us ever telling it what an edge or a curve is.


In [ ]:
from sklearn.model_selection import train_test_split

X = digits.images.astype(np.float32) / 16.0  # scale pixel values to 0..1
y = digits.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

X_train_t = torch.from_numpy(X_train).unsqueeze(1)  # add channel dimension -> (N, 1, 8, 8)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test).unsqueeze(1)
y_test_t = torch.from_numpy(y_test)

print("Training images:", X_train_t.shape)
print("Test images:", X_test_t.shape)


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

epochs = 30
batch_size = 32
n = X_train_t.size(0)

train_losses, test_accuracies = [], []

for epoch in range(epochs):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    model.train()
    for i in range(0, n, batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_train_t[idx], y_train_t[idx]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = loss_fn(outputs, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= n

    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).argmax(dim=1)
        accuracy = (preds == y_test_t).float().mean().item()

    train_losses.append(epoch_loss)
    test_accuracies.append(accuracy)

    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:2d}  |  loss {epoch_loss:.4f}  |  test accuracy {accuracy:.1%}")

print(f"\nFinal test accuracy: {test_accuracies[-1]:.1%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_losses)
axes[0].set_title("Training loss (lower = better)")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")

axes[1].plot([a * 100 for a in test_accuracies])
axes[1].set_title("Test accuracy (higher = better)")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy (%)")
plt.tight_layout()
plt.show()


## Step 5 — Did it actually learn?

Let's look at some test images the network has never seen before, along with its predictions. Green title = correct, red title = mistake.


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_test_t[:15])
    preds = logits.argmax(dim=1)

fig, axes = plt.subplots(3, 5, figsize=(11, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test_t[i, 0], cmap="gray")
    correct = preds[i].item() == y_test_t[i].item()
    color = "green" if correct else "red"
    ax.set_title(f"pred: {preds[i].item()}  (true: {y_test_t[i].item()})", color=color)
    ax.axis("off")
plt.suptitle("Network predictions on unseen test digits")
plt.tight_layout()
plt.show()


## Step 6 — Peeking inside: what did the network learn?

Remember our hand-made edge filter from Step 2? Let's see what filters the network *invented for itself* during training, and what a feature map looks like after training.


In [ ]:
learned_filters = model.conv1.weight.detach().numpy()  # shape (8, 1, 3, 3)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(learned_filters[i, 0], cmap="RdBu")
    ax.set_title(f"learned filter {i+1}")
    ax.axis("off")
plt.suptitle("The 8 filters the network learned on its own (compare to our hand-made edge filter!)")
plt.tight_layout()
plt.show()


In [ ]:
# Feed one test image through just the first conv layer and look at the resulting feature maps
model.eval()
with torch.no_grad():
    example = X_test_t[0:1]
    feature_maps = model.conv1(example)[0]  # shape (8, 8, 8)

fig, axes = plt.subplots(1, 9, figsize=(16, 2.5))
axes[0].imshow(example[0, 0], cmap="gray")
axes[0].set_title("input")
axes[0].axis("off")
for i in range(8):
    axes[i + 1].imshow(feature_maps[i], cmap="gray")
    axes[i + 1].set_title(f"map {i+1}")
    axes[i + 1].axis("off")
plt.suptitle("Input digit -> 8 feature maps produced by the learned filters")
plt.tight_layout()
plt.show()


## Wrap-up

You just watched a network go from random guessing (~10% accuracy, the same as chance with 10 digit classes) to recognizing handwritten digits with high accuracy — **without ever being told what an edge, curve, or loop looks like**. It discovered useful filters purely by adjusting numbers to reduce its mistakes.

**This is exactly the building block we'll reuse for Pong:**
- Instead of an 8x8 digit, the input will be a Pong game screen (many more pixels).
- Instead of "which digit is this?", the network will learn "which action (up / down / stay) leads to winning?"
- The convolution + pooling layers work the same way: they compress raw pixels into useful features (ball position, paddle position, direction of motion).
- Instead of a fixed label to learn from, the "correct answer" comes from **trial and error and rewards** — that's the reinforcement learning part we add on top of this CNN foundation.

### Ideas for using this with MBO students
- Let students **swap the edge_kernel** in Step 2 for other 3x3 grids (e.g. a blur filter `[[1,1,1],[1,1,1],[1,1,1]]/9`) and predict what will happen before running it.
- Have them **change the number of filters** (e.g. 8 -> 4 or 8 -> 16) and discuss the accuracy/speed trade-off.
- Ask them to **guess the label** of a raw pixel-number grid (Step 1) before revealing the picture — makes the "image = numbers" idea stick.
- Use the final feature-map picture as a discussion point: "which map looks like it's detecting the top of the digit? the bottom? a loop?"
